# Exploratory Data Analysis
Run this notebook FIRST to understand your data before training.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, '..')
from utils.data_loader import load_raw_data, standardise_columns, clean_data, resample_weekly

# ── Load ─────────────────────────────────────────────────────────────
df_raw = load_raw_data('../data/sales_data.xlsx')
print('Raw shape:', df_raw.shape)
df_raw.head()

In [ ]:
df = standardise_columns(df_raw)
df = clean_data(df)
print('After cleaning:', df.shape)
print('States:', df['state'].nunique())
print('Date range:', df['date'].min(), '→', df['date'].max())
df.isnull().sum()

In [ ]:
# Top 5 states by total sales
top5 = df.groupby('state')['sales'].sum().nlargest(5).index.tolist()
df_w = resample_weekly(df)

fig, axes = plt.subplots(5, 1, figsize=(14, 20))
for ax, state in zip(axes, top5):
    s = df_w[df_w['state'] == state].sort_values('date')
    ax.plot(s['date'], s['sales'])
    ax.set_title(state)
    ax.set_ylabel('Sales')
plt.tight_layout()
plt.savefig('../outputs/top5_states.png', dpi=120)
plt.show()

In [ ]:
# Monthly seasonality heatmap
df_w['month'] = pd.to_datetime(df_w['date']).dt.month
df_w['year']  = pd.to_datetime(df_w['date']).dt.year
pivot = df_w.groupby(['year','month'])['sales'].mean().unstack()
plt.figure(figsize=(12, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=.5)
plt.title('Average Weekly Sales by Year & Month')
plt.savefig('../outputs/seasonality_heatmap.png', dpi=120)
plt.show()